# Exp0.2 — Adaptive Refinement на реальных данных (CIFAR-10)

## Гипотеза
При одинаковом бюджете «дорогих операций» (refinement), выбор тайлов по локальной потере деталей (ρ)
даёт лучшее качество восстановления, чем случайный выбор тайлов.

## Протокол
1. Берём CIFAR-10, upscale до **128×128** (bilinear).
2. `coarse` = downsample ×4 → upsample обратно (теряем детали).
3. `refine` = итеративное приближение к GT на выбранных тайлах.
4. **Adaptive**: выбирает тайлы с наибольшим ρ (residual energy на grayscale).
5. **Baseline**: выбирает тайлы случайно.
6. Сравниваем MSE_rgb, PSNR_rgb, MSE_highfreq при равном бюджете.

## Загрузка CIFAR-10
Ноутбук пробует загрузить данные автоматически в таком порядке:
1. `torchvision.datasets.CIFAR10(download=True)`
2. `keras.datasets.cifar10.load_data()`
3. Локальный файл `./data/cifar-10-python.tar.gz`

**Ручной fallback:** если автоматическая загрузка не работает —
скачай архив вручную:
```
wget https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz -P ./data/
```
и перезапусти ноутбук.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import zoom, convolve
import warnings, time, os, pathlib, json

warnings.filterwarnings("ignore")

# ═══════════════════════════════════════════
# КОНФИГ
# ═══════════════════════════════════════════
H = 128
W = 128
tile_size = 16
downsample_factor = 4

alpha = 0.25
n_iter = 8

coverage_list = [0.01, 0.02, 0.05, 0.1, 0.2]

N_images_debug = 128
N_images_final = 2000

mode = "debug"  # "debug" или "final"

edge_eps = 1e-6
max_static_images = 10

save_images = False

# ─── Derived ───
N_images = N_images_debug if mode == "debug" else N_images_final
tiles_y = H // tile_size
tiles_x = W // tile_size
N_tiles = tiles_y * tiles_x

# Laplacian 3×3 kernel (fixed)
LAPLACIAN_K = np.array([[0, -1, 0],
                        [-1,  4, -1],
                        [0, -1, 0]], dtype=np.float32)

OUT_DIR = pathlib.Path("out/exp02")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Mode: {mode}, N_images: {N_images}")
print(f"Image size: {H}×{W}, tile: {tile_size}×{tile_size}")
print(f"Grid: {tiles_y}×{tiles_x} = {N_tiles} tiles")
print(f"Coverage list: {coverage_list}")
print(f"K per coverage: {[max(1, round(c * N_tiles)) for c in coverage_list]}")

# Save config
config = dict(H=H, W=W, tile_size=tile_size, downsample_factor=downsample_factor,
              alpha=alpha, n_iter=n_iter, coverage_list=coverage_list,
              N_images=N_images, mode=mode)
with open(OUT_DIR / "config.json", "w") as f:
    json.dump(config, f, indent=2)
print(f"Config saved to {OUT_DIR / 'config.json'}")


In [ ]:
def _load_cifar_torchvision():
    import torchvision
    ds = torchvision.datasets.CIFAR10(root="./data", train=True, download=True)
    # ds.data is uint8 (N, 32, 32, 3)
    return ds.data.astype(np.float32) / 255.0

def _load_cifar_keras():
    from keras.datasets import cifar10
    (x_train, _), (_, _) = cifar10.load_data()
    return x_train.astype(np.float32) / 255.0

def _load_cifar_pickle(path="./data/cifar-10-python.tar.gz"):
    import tarfile, pickle
    data_batches = []
    with tarfile.open(path, "r:gz") as tar:
        for member in tar.getmembers():
            if "data_batch" in member.name:
                f = tar.extractfile(member)
                batch = pickle.load(f, encoding="bytes")
                arr = batch[b"data"]  # (10000, 3072)
                arr = arr.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)  # (N, 32, 32, 3)
                data_batches.append(arr)
    data = np.concatenate(data_batches, axis=0)
    return data.astype(np.float32) / 255.0

# Try loading in order of preference
cifar_raw = None
for loader_name, loader_fn in [
    ("torchvision", _load_cifar_torchvision),
    ("keras", _load_cifar_keras),
    ("local pickle", lambda: _load_cifar_pickle("./data/cifar-10-python.tar.gz")),
]:
    try:
        print(f"Trying {loader_name}...", end=" ")
        cifar_raw = loader_fn()
        print(f"OK! Shape: {cifar_raw.shape}")
        break
    except Exception as e:
        print(f"Failed ({e})")

if cifar_raw is None:
    raise RuntimeError(
        "Could not load CIFAR-10. Download manually:\n"
        "  wget https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz -P ./data/\n"
        "and restart the notebook."
    )

print(f"CIFAR-10 loaded: {cifar_raw.shape}, dtype={cifar_raw.dtype}, "
      f"range=[{cifar_raw.min():.2f}, {cifar_raw.max():.2f}]")


In [ ]:
def upscale_image(img_32, target_h, target_w):
    """Upscale a 32×32×3 image to target_h×target_w×3 using bilinear interpolation."""
    # zoom factors per axis: (h_factor, w_factor, 1 for channels)
    fh = target_h / img_32.shape[0]
    fw = target_w / img_32.shape[1]
    up = zoom(img_32, (fh, fw, 1), order=1)  # order=1 = bilinear
    return np.clip(up[:target_h, :target_w, :], 0, 1).astype(np.float32)

# Subsample dataset and upscale
rng_data = np.random.RandomState(0)
indices = rng_data.choice(len(cifar_raw), size=N_images, replace=False)
indices.sort()

print(f"Upscaling {N_images} images from 32×32 to {H}×{W}...")
t0 = time.time()
images = np.stack([upscale_image(cifar_raw[i], H, W) for i in indices])
print(f"Done in {time.time()-t0:.1f}s. Shape: {images.shape}, "
      f"range=[{images.min():.3f}, {images.max():.3f}]")


In [ ]:
def coarse_from_image(img, s):
    """Downsample by factor s then bilinear upsample back. Works on H×W×3."""
    h, w, c = img.shape
    small = zoom(img, (1.0/s, 1.0/s, 1), order=1)
    big = zoom(small, (s, s, 1), order=1)
    return np.clip(big[:h, :w, :], 0, 1).astype(np.float32)

# Precompute all coarse images
print(f"Computing coarse (s={downsample_factor})...")
t0 = time.time()
coarses = np.stack([coarse_from_image(images[i], downsample_factor) for i in range(N_images)])
print(f"Done in {time.time()-t0:.1f}s.")

# Quick sanity check
mse_coarse_avg = np.mean((images - coarses)**2)
print(f"Mean coarse MSE across dataset: {mse_coarse_avg:.6f}")


In [ ]:
def to_grayscale(img):
    """Convert H×W×3 RGB to H×W grayscale using standard weights."""
    return (0.299 * img[:,:,0] + 0.587 * img[:,:,1] + 0.114 * img[:,:,2]).astype(np.float32)

def compute_tile_rho(img, coarse, tile_size):
    """ρ = residual energy on grayscale per tile.
    residual = Y_gt - Y_coarse; rho(tile) = mean(residual_tile^2)."""
    Y_gt = to_grayscale(img)
    Y_co = to_grayscale(coarse)
    residual = Y_gt - Y_co
    h, w = Y_gt.shape
    ty = h // tile_size
    tx = w // tile_size
    rho = np.zeros((ty, tx), dtype=np.float32)
    for i in range(ty):
        for j in range(tx):
            y0, y1 = i * tile_size, (i+1) * tile_size
            x0, x1 = j * tile_size, (j+1) * tile_size
            rho[i, j] = np.mean(residual[y0:y1, x0:x1] ** 2)
    return rho

# Test on first image
rho_test = compute_tile_rho(images[0], coarses[0], tile_size)
print(f"Rho shape: {rho_test.shape}, max: {rho_test.max():.6f}, min: {rho_test.min():.6f}")


In [ ]:
def select_tiles_adaptive(rho, K):
    """Top-K tiles by rho descending, tie-break by row-major flat_idx ascending."""
    ty, tx = rho.shape
    entries = []
    for i in range(ty):
        for j in range(tx):
            flat_idx = i * tx + j
            entries.append((-rho[i, j], flat_idx))  # neg rho for ascending sort
    entries.sort()
    selected = []
    for _, flat_idx in entries[:K]:
        i = flat_idx // tx
        j = flat_idx % tx
        selected.append((i, j))
    return selected

def select_tiles_random(rho, K, rng):
    """Random K tiles, no replacement."""
    ty, tx = rho.shape
    all_tiles = [(i, j) for i in range(ty) for j in range(tx)]
    idxs = rng.choice(len(all_tiles), size=min(K, len(all_tiles)), replace=False)
    return [all_tiles[idx] for idx in sorted(idxs)]


In [ ]:
def refine(pred, GT, selected_tiles, tile_size, n_iter, alpha):
    """Iterative refinement on selected tiles (H×W×3).
    pred_tile += alpha * (GT_tile - pred_tile), n_iter times.
    Returns copy. Cost = K * n_iter."""
    out = pred.copy()
    for _ in range(n_iter):
        for (i, j) in selected_tiles:
            y0, y1 = i * tile_size, (i+1) * tile_size
            x0, x1 = j * tile_size, (j+1) * tile_size
            out[y0:y1, x0:x1, :] += alpha * (GT[y0:y1, x0:x1, :] - out[y0:y1, x0:x1, :])
    return out


In [ ]:
def compute_metrics(img_gt, img_pred):
    """Compute MSE_rgb, PSNR_rgb, MSE_highfreq.

    MSE_rgb, PSNR_rgb: computed on all 3 channels.
    MSE_highfreq: Laplacian on grayscale, abs, normalized per image.
    """
    # ── RGB metrics ──
    mse_rgb = float(np.mean((img_gt - img_pred) ** 2))
    psnr_rgb = float(10 * np.log10(1.0 / max(mse_rgb, 1e-12)))

    # ── High-frequency metric (grayscale Laplacian) ──
    Y_gt = to_grayscale(img_gt)
    Y_pred = to_grayscale(img_pred)

    L_gt = np.abs(convolve(Y_gt, LAPLACIAN_K, mode='reflect'))
    L_pred = np.abs(convolve(Y_pred, LAPLACIAN_K, mode='reflect'))

    norm = np.mean(L_gt) + edge_eps
    L_gt_n = L_gt / norm
    L_pred_n = L_pred / norm

    mse_hf = float(np.mean((L_pred_n - L_gt_n) ** 2))

    return {
        "MSE_rgb": mse_rgb,
        "PSNR_rgb": psnr_rgb,
        "MSE_highfreq": mse_hf,
    }

# Test
m_test = compute_metrics(images[0], coarses[0])
print(f"Coarse metrics (image 0): {m_test}")


In [ ]:
results = []

t_start = time.time()
for img_idx in range(N_images):
    img = images[img_idx]
    coarse = coarses[img_idx]
    rho = compute_tile_rho(img, coarse, tile_size)

    # ── Coarse reference (coverage=0) ──
    m_coarse = compute_metrics(img, coarse)
    results.append({
        "image_idx": img_idx,
        "coverage": 0.0,
        "K": 0,
        "budget": 0,
        "method": "coarse",
        **m_coarse,
    })

    for cov in coverage_list:
        K = max(1, round(cov * N_tiles))

        # ── Adaptive ──
        sel_adaptive = select_tiles_adaptive(rho, K)
        pred_adaptive = refine(coarse.copy(), img, sel_adaptive, tile_size, n_iter, alpha)
        m_adaptive = compute_metrics(img, pred_adaptive)

        # ── Baseline (random) ──
        # RNG seed: image_idx * 10000 + int(cov * 10000) — unique per image × coverage
        rng_bl = np.random.RandomState(img_idx * 10000 + int(cov * 10000))
        sel_baseline = select_tiles_random(rho, K, rng_bl)
        pred_baseline = refine(coarse.copy(), img, sel_baseline, tile_size, n_iter, alpha)
        m_baseline = compute_metrics(img, pred_baseline)

        budget = K * n_iter

        for method, m in [("adaptive", m_adaptive), ("baseline", m_baseline)]:
            results.append({
                "image_idx": img_idx,
                "coverage": cov,
                "K": K,
                "budget": budget,
                "method": method,
                **m,
            })

    if (img_idx + 1) % max(1, N_images // 10) == 0:
        print(f"  {img_idx+1}/{N_images} images processed...")

elapsed = time.time() - t_start
df = pd.DataFrame(results)
print(f"\nExperiment done in {elapsed:.1f}s — {len(df)} rows")
df.head(10)


In [ ]:
# Showcase: 10 images, coverage=5% (a useful middle ground)
showcase_coverage = 0.05
showcase_K = max(1, round(showcase_coverage * N_tiles))
n_showcase = min(max_static_images, N_images)

# Pick evenly spaced images for diversity
showcase_indices = np.linspace(0, N_images - 1, n_showcase, dtype=int)

for si, img_idx in enumerate(showcase_indices):
    img = images[img_idx]
    coarse = coarses[img_idx]
    rho = compute_tile_rho(img, coarse, tile_size)

    sel_a = select_tiles_adaptive(rho, showcase_K)
    rng_bl = np.random.RandomState(img_idx * 10000 + int(showcase_coverage * 10000))
    sel_b = select_tiles_random(rho, showcase_K, rng_bl)

    pred_a = refine(coarse.copy(), img, sel_a, tile_size, n_iter, alpha)
    pred_b = refine(coarse.copy(), img, sel_b, tile_size, n_iter, alpha)

    # Masks for visualization
    mask_a = np.zeros((tiles_y, tiles_x), dtype=float)
    for (i, j) in sel_a: mask_a[i, j] = 1.0
    mask_b = np.zeros((tiles_y, tiles_x), dtype=float)
    for (i, j) in sel_b: mask_b[i, j] = 1.0

    # Error maps (grayscale magnitude)
    err_c = np.sqrt(np.mean((img - coarse)**2, axis=2))
    err_b = np.sqrt(np.mean((img - pred_b)**2, axis=2))
    err_a = np.sqrt(np.mean((img - pred_a)**2, axis=2))
    err_max = max(err_c.max(), err_b.max(), err_a.max(), 1e-6)

    fig, axes = plt.subplots(2, 5, figsize=(22, 9))
    fig.suptitle(f"Image #{img_idx} | coverage={showcase_coverage} (K={showcase_K})", fontsize=13)

    row0 = [
        ("GT (original)", img),
        ("Coarse", coarse),
        ("Baseline pred", pred_b),
        ("Adaptive pred", pred_a),
        ("|err| coarse", err_c),
    ]
    for ax, (title, data) in zip(axes[0], row0):
        if data.ndim == 3:
            ax.imshow(np.clip(data, 0, 1))
        else:
            ax.imshow(data, cmap="magma", vmin=0, vmax=err_max)
        ax.set_title(title, fontsize=10)
        ax.axis("off")

    row1 = [
        ("|err| baseline", err_b),
        ("|err| adaptive", err_a),
        ("ρ heatmap", rho),
        ("baseline mask", mask_b),
        ("adaptive mask", mask_a),
    ]
    for ax, (title, data) in zip(axes[1], row1):
        if "err" in title:
            ax.imshow(data, cmap="magma", vmin=0, vmax=err_max)
        elif "ρ" in title or "rho" in title.lower():
            ax.imshow(data, cmap="hot", interpolation="nearest")
        else:
            ax.imshow(data, cmap="hot", interpolation="nearest")
        ax.set_title(title, fontsize=10)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

print(f"Showcase: {n_showcase} images displayed")


In [ ]:
# Plots: MSE_rgb, MSE_highfreq, PSNR_rgb vs coverage
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("CIFAR-10 Adaptive vs Baseline (mean over dataset)", fontsize=14)

for ax, metric in zip(axes, ["MSE_rgb", "MSE_highfreq", "PSNR_rgb"]):
    # Coarse reference (coverage=0)
    coarse_val = df[df["method"] == "coarse"][metric].mean()
    ax.axhline(coarse_val, color="gray", linestyle="--", linewidth=1, label="coarse (no refine)")

    for method, color, marker in [("adaptive", "tab:blue", "o"), ("baseline", "tab:orange", "s")]:
        sub = df[(df["method"] == method) & (df["coverage"] > 0)]
        grp = sub.groupby("coverage")[metric]
        mean_ = grp.mean()
        std_ = grp.std()
        ax.errorbar(mean_.index, mean_.values, yerr=std_.values,
                    label=method, color=color, marker=marker, capsize=3)
    ax.set_xlabel("Coverage")
    ax.set_ylabel(metric)
    ax.set_title(metric)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# ═══════════════════════════════════════════
# Агрегирование и финальный вердикт
# ═══════════════════════════════════════════

# ── Coarse reference ──
print("=== COARSE REFERENCE (coverage=0) ===")
coarse_ref = df[df["method"] == "coarse"][["MSE_rgb", "MSE_highfreq", "PSNR_rgb"]].mean()
print(coarse_ref.round(6).to_string())
print()

# ── Adaptive vs Baseline ──
summary_rows = []
for cov in coverage_list:
    sub = df[df["coverage"] == cov]
    a = sub[sub["method"] == "adaptive"]
    b = sub[sub["method"] == "baseline"]
    for metric in ["MSE_rgb", "MSE_highfreq", "PSNR_rgb"]:
        a_mean = a[metric].mean()
        a_std = a[metric].std()
        b_mean = b[metric].mean()
        b_std = b[metric].std()
        if metric == "PSNR_rgb":
            wins = a_mean > b_mean
            delta_pct = (a_mean - b_mean) / max(abs(b_mean), 1e-12) * 100
        else:
            wins = a_mean < b_mean
            delta_pct = (b_mean - a_mean) / max(abs(b_mean), 1e-12) * 100
        summary_rows.append({
            "coverage": cov,
            "metric": metric,
            "adaptive_mean": round(a_mean, 7),
            "adaptive_std": round(a_std, 7),
            "baseline_mean": round(b_mean, 7),
            "baseline_std": round(b_std, 7),
            "delta_%": round(delta_pct, 2),
            "adaptive_wins": wins,
        })

summary = pd.DataFrame(summary_rows)
print("=== ADAPTIVE vs BASELINE ===")
print(summary.to_string(index=False))

# ── Per-image win rate ──
print("\n=== PER-IMAGE WIN RATE (adaptive MSE_rgb < baseline MSE_rgb) ===")
for cov in coverage_list:
    sub = df[df["coverage"] == cov].pivot_table(
        index="image_idx", columns="method", values="MSE_rgb")
    if "adaptive" in sub.columns and "baseline" in sub.columns:
        per_img_wins = (sub["adaptive"] < sub["baseline"]).mean() * 100
        print(f"  coverage={cov:.0%}: adaptive wins on {per_img_wins:.1f}% of images")

# ── Verdict ──
n_comparisons = len(summary)
n_wins = summary["adaptive_wins"].sum()
win_rate = n_wins / n_comparisons * 100

hf_summary = summary[summary["metric"] == "MSE_highfreq"]
hf_wins = hf_summary["adaptive_wins"].sum()
hf_total = len(hf_summary)

print(f"\n{'='*60}")
print(f"Overall: adaptive wins {n_wins}/{n_comparisons} ({win_rate:.0f}%)")
print(f"MSE_highfreq: adaptive wins {hf_wins}/{hf_total}")

if win_rate >= 75 and (hf_total == 0 or hf_wins / max(hf_total, 1) >= 0.7):
    verdict = "✅ ГИПОТЕЗА ПОДТВЕРЖДЕНА — adaptive refinement работает на реальных данных"
else:
    verdict = "❌ ГИПОТЕЗА НЕ ПОДТВЕРЖДЕНА"

print(f"\n>>> {verdict}")
print(f"{'='*60}")


In [ ]:
df.to_csv(OUT_DIR / "results.csv", index=False)
summary.to_csv(OUT_DIR / "summary.csv", index=False)
print(f"Saved: {OUT_DIR / 'results.csv'}, {OUT_DIR / 'summary.csv'}")


In [ ]:
# ═══════════════════════════════════════════
# REPRO-CHECK
# ═══════════════════════════════════════════
print("Running repro-check...")
repro_idx = 0
repro_cov = 0.05

for run in range(2):
    img_r = images[repro_idx]
    coarse_r = coarses[repro_idx]
    rho_r = compute_tile_rho(img_r, coarse_r, tile_size)

    K_r = max(1, round(repro_cov * N_tiles))
    sel_r = select_tiles_adaptive(rho_r, K_r)
    pred_r = refine(coarse_r.copy(), img_r, sel_r, tile_size, n_iter, alpha)
    m_r = compute_metrics(img_r, pred_r)

    if run == 0:
        sel_first = sel_r
        m_first = m_r
    else:
        assert sel_r == sel_first, "REPRO FAIL: tile selection differs!"
        for key in m_first:
            v1, v2 = m_first[key], m_r[key]
            if np.isnan(v1) and np.isnan(v2):
                continue
            assert abs(v1 - v2) < 1e-10, f"REPRO FAIL: {key} differs: {v1} vs {v2}"

print("✅ Repro-check PASSED: tile selection and metrics are identical across runs.")
